In [1]:
import sys
sys.path.append("../src")        # lets the notebook import from src/

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from features import clean_data, add_features, build_pipeline, NUMERIC, CATEGORICAL

In [2]:
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = clean_data(df)

y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn", "customerID"])

print(X.shape, y.shape)

(7043, 19) (7043,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Churn rate train:", round(y_train.mean(), 3))
print("Churn rate test: ", round(y_test.mean(), 3))

Train: (5634, 19) Test: (1409, 19)
Churn rate train: 0.265
Churn rate test:  0.265


In [4]:
X_new = add_features(X_train)
X_new[["tenure", "TotalCharges", "num_addons", "is_autopay",
       "avg_charge_per_month", "tenure_group"]].head(8)

,tenure,TotalCharges,num_addons,is_autopay,avg_charge_per_month,tenure_group
3738,35,1701.65,3,0,48.618571,25-48
3151,15,1151.55,1,0,76.770000,13-24
4860,13,590.35,3,0,45.411538,13-24
3867,26,1905.70,4,1,73.296154,25-48
3810,1,44.55,0,0,44.550000,0-12
2666,66,5196.10,5,1,78.728788,49-72
2645,1,19.80,0,1,19.800000,0-12
189,40,2379.10,1,1,59.477500,25-48


In [5]:
check = X_new.copy()
check["Churn"] = y_train

print(check.groupby("num_addons")["Churn"].mean().round(3), "\n")
print(check.groupby("is_autopay")["Churn"].mean().round(3), "\n")
print(check.groupby("tenure_group")["Churn"].mean().round(3))

num_addons
0    0.209
1    0.458
2    0.362
3    0.278
4    0.230
5    0.125
6    0.047
Name: Churn, dtype: float64 

is_autopay
0    0.350
1    0.155
Name: Churn, dtype: float64 

tenure_group
0-12     0.473
13-24    0.286
25-48    0.202
49-72    0.099
Name: Churn, dtype: float64


In [6]:
pipe = build_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"))
pipe.fit(X_train, y_train)
print("Pipeline trained")

Pipeline trained


In [7]:
print(pipe.named_steps.keys())

feature_names = pipe.named_steps["prep"].get_feature_names_out()
print("Number of model inputs:", len(feature_names))
print(feature_names[:10])

dict_keys(['features', 'prep', 'model'])
Number of model inputs: 53
['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges' 'num__num_addons'
 'num__is_autopay' 'num__avg_charge_per_month' 'cat__gender_Female'
 'cat__gender_Male' 'cat__SeniorCitizen_No' 'cat__SeniorCitizen_Yes']


In [8]:
proba = pipe.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

Test ROC-AUC: 0.842


In [9]:
new_customer = pd.DataFrame([{
    "gender": "Female", "SeniorCitizen": "No", "Partner": "No", "Dependents": "No",
    "tenure": 2, "PhoneService": "Yes", "MultipleLines": "No",
    "InternetService": "Fiber optic", "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No", "StreamingTV": "No",
    "StreamingMovies": "No", "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check", "MonthlyCharges": 80.0, "TotalCharges": 160.0,
}])

print("Churn probability:", round(pipe.predict_proba(new_customer)[0, 1], 3))

Churn probability: 0.848


## Pipeline summary
- Split: 80/20, stratified, random_state=42 (train 5,634 / test 1,409).
- New features: num_addons, is_autopay, avg_charge_per_month, tenure_group.
- Pipeline: add_features -> StandardScaler (numeric) + OneHotEncoder (categorical) -> model.
- Everything is fitted on the training data only, so there is no data leakage.
- Sanity check: Logistic Regression test ROC-AUC = 0.8X.